In [1]:
# import all libraries needed
import json
import re
import pandas as pd
from sklearn.metrics import classification_report
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download("vader_lexicon")

# import data and get only data with annotations
with open("../../01_data/annotations_reduced.json", "r") as f:
    data = json.load(f)

data_with_annotations = []
for task in data:
    if task["annotations"]:
        data_with_annotations.append(task)

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/maxweiland/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [2]:
# create instance of VADER sentiment analyzer
sid = SentimentIntensityAnalyzer()

def stance_vader_baseline(sentence, span_text, sid, whole_sentence=False, window=6):

    # just select the whole sentence if parameter is specified as such
    if whole_sentence:
        compound_score = sid.polarity_scores(sentence)["compound"]
    
    # otherwise, apply window-based sentiment analysis
    else:
        # get all tokens in the entire sentence and in the span
        sentence_tokens = re.findall(r"\w+|'\w+|[^\w\s]", sentence)
        span_tokens = re.findall(r"\w+|'\w+|[^\w\s]", span_text)

        # get the indices of the first occurrence of the span
        span_indices = []
        for i in range(len(sentence_tokens) - len(span_tokens) + 1):
            if sentence_tokens[i : i + len(span_tokens)] == span_tokens:
                span_indices.append(i)
                span_indices.append(i + len(span_tokens) - 1)
        
        # extract the context around this first occurrence
        if span_indices:
            start_span = span_indices[0]
            end_span = span_indices[1]
            start = max(0, start_span - window)
            end = min(len(sentence_tokens), end_span + window + 1)
            context_text = " ".join(sentence_tokens[start:end])
        else:
            # fallback to whole sentence
            context_text = sentence
        
        # Get VADER sentiment scores
        compound_score = sid.polarity_scores(context_text)["compound"]

    # return sentiment score based on common threshold
    if compound_score > 0.5:
        return "pos"
    elif compound_score < -0.5:
        return "neg"
    else:
        return "neutral"

In [3]:
# find the best window size based on macro f1 score
best_window = 0
best_macro = 0

for window_size in range(1, 20):
    true_labels, pred_labels = [], []

    for item in data:
        sentence = item["sentence"]
        for ann in item["annotations"]:
            span_text = ann["text"]
            true_labels.append(ann["tag"][3:])
            pred = stance_vader_baseline(sentence, span_text, sid, whole_sentence=False, window=window_size)
            pred_labels.append(pred)
    macro_f1 = classification_report(true_labels, pred_labels, output_dict=True)["macro avg"]["f1-score"]
    if macro_f1 > best_macro:
        best_window = window_size
        best_macro = macro_f1

print(f"Best macro f1 score of {best_macro:.4f} achieved with window size of {best_window}")

true_labels, pred_labels = [], []

for item in data:
    sentence = item["sentence"]
    for ann in item["annotations"]:
        span_text = ann["text"]
        true_labels.append(ann["tag"][3:])
        pred = stance_vader_baseline(sentence, span_text, sid, whole_sentence=True, window=6)
        pred_labels.append(pred)
macro_f1_full_sentence = classification_report(true_labels, pred_labels, output_dict=True)["macro avg"]["f1-score"]
print(f"Using always the full sentence gives macro f1 score of {macro_f1_full_sentence:.4f}")

Best macro f1 score of 0.3701 achieved with window size of 14
Using always the full sentence gives macro f1 score of 0.3790


In [6]:
test_metrics = {}
for item in data:
    sentence = item["sentence"]
    for ann in item["annotations"]:
        span_text = ann["text"]
        true_labels.append(ann["tag"][3:])
        pred = stance_vader_baseline(sentence, span_text, sid, whole_sentence=True, window=6)
        pred_labels.append(pred)
metrics = classification_report(true_labels, pred_labels, output_dict=True)
test_metrics["dictionary_baseline"] = {
    "negative": metrics["neg"]["f1-score"],
    "neutral": metrics["neutral"]["f1-score"],
    "positive": metrics["pos"]["f1-score"]
    }
with open("evaluation_metrics_dictionary.json", "w") as f:
    json.dump(test_metrics, f, indent=4)